# Mandatory assignment 3

**GitHub repository:** *https://github.com/FilipThue/TECH2-H26-assignment3*

## About the assignment

- Complete the assignment individually.
- You may use online resources, including generative AI. Include a statement
  explaining how you used AI, or state that you did not use it.

## Submission

- Submit your completed `assignment3.ipynb` on Canvas by
  **October 23, 2026, at 23:59**.

## Submission requirements

- Create a new repository from the template linked on Canvas (click **Use this
  template** > **Create a new repository**) and implement your solution in your
  GitHub repository.
- Use the TECH2 environment from the course.
- Implement your solution directly in this notebook by inserting code and text
  cells. Do not create additional notebooks or files.
- Include all requested numerical results, figures, and short written answers.
- Give each figure a title, label its axes, and include a legend where needed.
- Before you submit, make sure your GitHub repository is **publicly accessible** and include its URL at the
  top of this notebook.
- Before submission, restart the kernel and run all cells in order. Your notebook
  must run without errors and find the CSV without an absolute file path.

## Hints

- Use the material from lectures and workshops 1–3: functions, loops, NumPy,
  pandas, and Matplotlib.
- Compute group statistics using loops; there is no need to use `groupby()`
  (which we cover in lecture 4), but you can do so if you prefer.

***
## Stock market participation in US households

Who participates in the stock market, and how has participation changed over time?
To answer this question, you will analyze data from the
[Survey of Consumer Finances](https://www.federalreserve.gov/econres/scfindex.htm)
(SCF), a representative survey of US households conducted every three years
from 1989 to 2022.

The file `SCF_10pct.csv` contains a 10% sample of the SCF data. The central variables of
interest for this assignment are:

1. `stocks`, which reports the value of directly held stocks (in US dollars); and
2. `equity`, which reports the value of all equity holdings (in US dollars),
   including direct stock ownership and indirect exposure through mutual funds,
   retirement accounts, and other investment vehicles.

We measure stock market participation in two ways:

- **Direct participation:** a household owns individual stocks (`stocks > 0`).
- **Overall participation:** a household holds equity directly or indirectly
  (`equity > 0`).

For each measure, the *participation rate* is the percentage of households
satisfying that condition. We therefore calculate two rates: the **direct
participation rate** and the **overall participation rate**. Direct participants
are included among overall participants, so the two rates should not be added.

## Data description

| Variable | Description |
|----------|-------------|
| `id` | Unique observation identifier |
| `year` | Survey year |
| `age` | Age of the household reference person |
| `educ` | Education level: `1`: No high school; `2`: High school; `3`: Some college; `4`: Bachelor's or higher |
| `stocks` | Value of directly held individual stocks (in US dollars) |
| `equity` | Total directly and indirectly owned stock equity (in US dollars) |

***
## Part 1 — Data preparation

1. Load `SCF_10pct.csv` into a pandas DataFrame. Print the columns contained in
   the DataFrame and their number of non-missing values.
2. Drop all observations that have missing values for any of the variables listed
   in the table above. For each variable, report how many observations you dropped
   and how many remain.
3. Keep observations with `25 <= age <= 89`, including both endpoints.
   Report how many observations you dropped and how many remain.
4. Create two 0/1 indicator columns:
   - `direct`: 1 if the household participates directly (`stocks > 0`),
     0 otherwise.
   - `overall`: 1 if the household participates directly or indirectly
     (`equity > 0`), 0 otherwise.

In [57]:
import pandas as pd

df_raw = pd.read_csv("SCF_10pct.csv", delimiter = ",", index_col = "id")

#2
dropped_observations = df_raw.count()
df_raw = df_raw.dropna(axis = 0)
dropped_observations -= df_raw.count()
print(f'Amount of dropped observations:\n{dropped_observations}')
print()
print(f'Amount of remaining observations:\n{df_raw.count()}')


Amount of dropped observations:
year      50
age       50
educ      34
stocks    32
equity    34
dtype: int64

Amount of remaining observations:
year      5758
age       5758
educ      5758
stocks    5758
equity    5758
dtype: int64


In [58]:
#3
dropped_ages = df_raw.count()
mask = (25 <= df_raw['age']) & (df_raw['age'] <= 89)
df = df_raw[mask]
dropped_ages -= df.count()
print(f'The amount of dropped observations due to age: {dropped_ages['age']}')
print(f'The amount of remaining observations: {df.count()['age']}')


The amount of dropped observations due to age: 326
The amount of remaining observations: 5432


In [60]:
df['direct'] = (df['stocks'] > 0).astype(int)
df['overall'] = (df['equity'] > 0).astype(int)
df


,year,age,educ,stocks,equity,direct,overall
id,,,,,,,
0,1989,35,2.0,0.0,0.0,0,0
1,1989,33,2.0,0.0,0.0,0,0
2,1989,62,1.0,0.0,0.0,0,0
4,1989,70,1.0,0.0,0.0,0,0
5,1989,50,1.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...
5802,2022,77,3.0,0.0,0.0,0,0
5804,2022,33,3.0,0.0,0.0,0,0
5805,2022,81,2.0,100000.0,100000.0,1,1


***
## Part 2 — Participation rates

You are now ready to compute the stock market participation rates for the
final sample. You will compute these in two ways: first, by writing a function
that uses a loop to compute the participation rate, and second, by using pandas's
built-in `mean()` method.

1. Write a function `participation_rate(indicator)` that takes a nonempty
   one-dimensional sequence of 0/1 values and returns the fraction of ones
   (in percent).
2. Apply your function to the final sample's `direct` and `overall`
   columns to compute the direct and overall participation rates. Report the
   results.
3. Use pandas's built-in `mean()` method to compute the direct and overall
   participation rates. Report the results.

In [73]:
#1
def participation_rate(indicator):
    counter = 0
    for value in indicator:
        if value == 1:
            counter += 1
    return counter/indicator.count() * 100

#2
direct_participation_rate = participation_rate(df['direct'])
overall_participation_rate = participation_rate(df['overall'])

print(f'The direct participation rate is: {round(direct_participation_rate, 2)}%')
print(f'The overall participation rate is: {round(overall_participation_rate, 2)}%')

print(f'The direct participation rate (pandas) is: {round(df['direct'].mean()*100, 2)}%')
print(f'The overall participation rate (pandas) is: {round(df['overall'].mean()*100, 2)}%')

The direct participation rate is: 16.96%
The overall participation rate is: 49.41%
The direct participation rate (pandas) is: 16.96%
The overall participation rate (pandas) is: 49.41%


***
## Part 3 — Participation by education

1. Loop over education categories 1–4. For each category, calculate the direct
   participation rate and the overall participation rate (in percent).
2. Display the results, rounding rates to one decimal place.
3. Create a figure with two subplots, one for each rate, containing bar charts
   of participation rates across education categories.
4. In two or three sentences, describe the education pattern.

In [ ]:
#1
def ed_participation_rate(indicator):
    education_participation = [0, 0, 0, 0]
    for i, value in enumerate(indicator):
        if value == 1:
            education_participation[(df['educ'].iloc[i]).astype(int)-1] += 1 / df['educ'].count() *100
    return education_participation

ed_direct_rate = ed_participation_rate(df['direct'])
ed_overall_rate = ed_participation_rate(df['overall'])

#2
for i in range(len(ed_direct_rate)):
    print(f'The direct participation level for education level {i+1} is {round(ed_direct_rate[i], 1)}, and the overall participation level is {round(ed_overall_rate[i], 1)}')





The direct participation level for education level 1 is 0.6, and the overall participation level is 2.4
The direct participation level for education level 2 is 3.3, and the overall participation level is 11.7
The direct participation level for education level 3 is 3.4, and the overall participation level is 12.2
The direct participation level for education level 4 is 9.6, and the overall participation level is 23.1


***
## Part 4 — Participation over time

1. Loop over years. For each year, compute the direct participation rate and the
   overall participation rate.
2. Create a line plot showing both participation rates over time in a single figure.
3. Report the changes from 1989 to 2022 in the direct and overall participation
   rates, both in percentage points.
4. Describe the main time patterns in two or three sentences.

***
## AI-use statement

1.3 - I asked AI if I could use True/False as instead of 0/1, it told me to include `.astype(int)`

3.1 - I asked AI when i wrote `df['educ'][i]`, and didnt understand why it didn't work. It said that it was searching by the index name and not the position, so i had to write `df['educ'].iloc[i]`